# Buddy Planner Colab Training

This notebook is the canonical Colab workflow for longer Buddy planner runs. It is designed for A100 runtimes with Drive-backed outputs, resumable checkpoints, and easy model/dataset overrides.


In [ ]:
!pip install -q transformers accelerate peft bitsandbytes sentencepiece safetensors pyyaml huggingface_hub


In [ ]:
import os
from getpass import getpass

HF_TOKEN = ""  # leave blank to skip; set or use getpass below
USE_GETPASS_FOR_HF_TOKEN = False

if USE_GETPASS_FOR_HF_TOKEN and not HF_TOKEN:
    HF_TOKEN = getpass("HF_TOKEN: ")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token configured.")
else:
    print("HF token not configured; continuing unauthenticated.")


In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Google Drive mount.")


In [ ]:
from pathlib import Path
import shutil

WORK_ROOT = Path("/content")
BUNDLE_ARCHIVE_NAME = "planner_colab_bundle.zip"
BUNDLE_ARCHIVE_PATH = WORK_ROOT / BUNDLE_ARCHIVE_NAME
EXTRACT_DIR = WORK_ROOT / "planner_colab_bundle"
DEFAULT_RUNS_ROOT = Path("/content/drive/MyDrive/buddy_planner_runs") if USE_DRIVE else WORK_ROOT / "buddy_planner_runs"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DEFAULT_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print({
    "work_root": str(WORK_ROOT),
    "bundle_archive": str(BUNDLE_ARCHIVE_PATH),
    "extract_dir": str(EXTRACT_DIR),
    "default_runs_root": str(DEFAULT_RUNS_ROOT),
})


In [ ]:
BUNDLE_SOURCE_MODE = "upload"  # "upload" or "drive"
BUNDLE_DRIVE_PATH = ""  # e.g. /content/drive/MyDrive/planner_colab_bundle.zip

if BUNDLE_SOURCE_MODE == "upload":
    get_ipython().run_line_magic("cd", "/content")
    from google.colab import files
    uploaded = files.upload()
    assert BUNDLE_ARCHIVE_NAME in uploaded, f"Upload {BUNDLE_ARCHIVE_NAME}."
else:
    assert BUNDLE_DRIVE_PATH, "Set BUNDLE_DRIVE_PATH when using drive mode."
    shutil.copy2(BUNDLE_DRIVE_PATH, BUNDLE_ARCHIVE_PATH)
    print(f"Copied bundle from {BUNDLE_DRIVE_PATH} to {BUNDLE_ARCHIVE_PATH}")


In [ ]:
get_ipython().system("rm -rf /content/planner_colab_bundle")
get_ipython().system("unzip -q /content/planner_colab_bundle.zip -d /content/planner_colab_bundle")
get_ipython().run_line_magic("cd", "/content/planner_colab_bundle")

from pathlib import Path

trainer_text = Path("/content/planner_colab_bundle/train_planner_model.py").read_text()
checks = {
    "has_dtype_kwarg": 'model_load_kwargs["dtype"] = dtype' in trainer_text,
    "uses_warmup_steps": '"warmup_steps": warmup_steps' in trainer_text,
    "filters_trainer_kwargs": 'Trainer(**_filter_supported_kwargs(Trainer, trainer_kwargs))' in trainer_text,
    "supports_resume": '--resume-from-checkpoint' in trainer_text,
    "supports_save_steps": '--save-steps' in trainer_text,
    "supports_lora_overrides": '--lora-r' in trainer_text,
}
print(checks)
assert all(checks.values()), "Uploaded bundle is stale; regenerate and re-upload planner_colab_bundle.zip from the repo."


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
RUN_PROFILES = {
    "a100_fast": {
        "max_steps": 300,
        "train_batch": 48,
        "grad_accum": 1,
        "warmup_steps": 20,
        "save_steps": 100,
        "save_total_limit": 3,
        "learning_rate": 2.0e-5,
        "max_seq_length": 1024,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
    },
    "a100_long": {
        "max_steps": 800,
        "train_batch": 48,
        "grad_accum": 1,
        "warmup_steps": 40,
        "save_steps": 100,
        "save_total_limit": 4,
        "learning_rate": 1.5e-5,
        "max_seq_length": 1024,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    "a100_2h": {
        "max_steps": 1200,
        "train_batch": 48,
        "grad_accum": 1,
        "warmup_steps": 60,
        "save_steps": 100,
        "save_total_limit": 6,
        "learning_rate": 1.2e-5,
        "max_seq_length": 1024,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    "a100_3b_long": {
        "max_steps": 600,
        "train_batch": 32,
        "grad_accum": 1,
        "warmup_steps": 40,
        "save_steps": 100,
        "save_total_limit": 4,
        "learning_rate": 1.2e-5,
        "max_seq_length": 1024,
        "gradient_checkpointing": false,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    "a100_3b_2h": {
        "max_steps": 900,
        "train_batch": 40,
        "grad_accum": 1,
        "warmup_steps": 60,
        "save_steps": 100,
        "save_total_limit": 6,
        "learning_rate": 1.0e-5,
        "max_seq_length": 1024,
        "gradient_checkpointing": false,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
}

PRESET = "a100"
RUN_PROFILE = "a100_long"  # a100_fast, a100_long, a100_2h, a100_3b_long, a100_3b_2h
BASE_MODEL_OVERRIDE = "Qwen/Qwen2.5-1.5B-Instruct"  # set Qwen/Qwen2.5-3B-Instruct when using a100_3b_* profiles
OUTPUT_ROOT = str(DEFAULT_RUNS_ROOT)
RUN_NAME = "planner_a100_long"
RESUME_FROM_CHECKPOINT = ""  # set to a checkpoint dir to resume explicitly
CUSTOM_TRAIN_PATH = ""  # optional: drive/uploaded train.jsonl path
CUSTOM_VALID_PATH = ""  # optional: drive/uploaded valid.jsonl path
DOWNLOAD_FINAL_ZIP = False
EXTRA_TRAIN_ARGS = []

profile = dict(RUN_PROFILES[RUN_PROFILE])
MAX_STEPS = profile["max_steps"]
LOGGING_STEPS = 1
TRAIN_BATCH_OVERRIDE = profile["train_batch"]
GRAD_ACCUM_OVERRIDE = profile["grad_accum"]
WARMUP_STEPS_OVERRIDE = profile["warmup_steps"]
SAVE_STEPS_OVERRIDE = profile["save_steps"]
SAVE_TOTAL_LIMIT_OVERRIDE = profile["save_total_limit"]
LEARNING_RATE_OVERRIDE = profile["learning_rate"]
MAX_SEQ_LENGTH_OVERRIDE = profile["max_seq_length"]
GRADIENT_CHECKPOINTING_OVERRIDE = profile.get("gradient_checkpointing")
LORA_R_OVERRIDE = profile["lora_r"]
LORA_ALPHA_OVERRIDE = profile["lora_alpha"]
LORA_DROPOUT_OVERRIDE = profile["lora_dropout"]


In [ ]:
from pathlib import Path
import json
import shlex

RUNS_ROOT = Path(OUTPUT_ROOT).expanduser()
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / "train.log"
TRAINER_CONFIG_PATH = Path("/content/planner_colab_bundle/planner_training_colab.yaml")

def latest_checkpoint(run_dir: Path):
    checkpoints = sorted(
        run_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    return checkpoints[-1] if checkpoints else None

auto_resume = latest_checkpoint(RUN_DIR) if not RESUME_FROM_CHECKPOINT else Path(RESUME_FROM_CHECKPOINT)
resume_path = str(auto_resume) if auto_resume else ""

cmd = [
    "python",
    "-u",
    "train_planner_model.py",
    "--config",
    str(TRAINER_CONFIG_PATH),
    "--skip-prepare",
    "--output-dir",
    str(RUN_DIR),
    "--colab-preset",
    PRESET,
    "--max-steps",
    str(MAX_STEPS),
    "--logging-steps",
    str(LOGGING_STEPS),
    "--no-torch-compile",
    "--per-device-train-batch-size",
    str(TRAIN_BATCH_OVERRIDE),
    "--gradient-accumulation-steps",
    str(GRAD_ACCUM_OVERRIDE),
    "--warmup-steps",
    str(WARMUP_STEPS_OVERRIDE),
    "--save-steps",
    str(SAVE_STEPS_OVERRIDE),
    "--save-total-limit",
    str(SAVE_TOTAL_LIMIT_OVERRIDE),
    "--learning-rate",
    str(LEARNING_RATE_OVERRIDE),
    "--max-seq-length",
    str(MAX_SEQ_LENGTH_OVERRIDE),
    "--lora-r",
    str(LORA_R_OVERRIDE),
    "--lora-alpha",
    str(LORA_ALPHA_OVERRIDE),
    "--lora-dropout",
    str(LORA_DROPOUT_OVERRIDE),
]

if GRADIENT_CHECKPOINTING_OVERRIDE is True:
    cmd.append("--gradient-checkpointing")
elif GRADIENT_CHECKPOINTING_OVERRIDE is False:
    cmd.append("--no-gradient-checkpointing")
if BASE_MODEL_OVERRIDE:
    cmd.extend(["--base-model", BASE_MODEL_OVERRIDE])
if CUSTOM_TRAIN_PATH:
    cmd.extend(["--train-path", CUSTOM_TRAIN_PATH])
if CUSTOM_VALID_PATH:
    cmd.extend(["--valid-path", CUSTOM_VALID_PATH])
if resume_path:
    cmd.extend(["--resume-from-checkpoint", resume_path])
if EXTRA_TRAIN_ARGS:
    cmd.extend(EXTRA_TRAIN_ARGS)

run_plan = {
    "preset": PRESET,
    "run_profile": RUN_PROFILE,
    "run_dir": str(RUN_DIR),
    "resume_from_checkpoint": resume_path or None,
    "custom_train_path": CUSTOM_TRAIN_PATH or None,
    "custom_valid_path": CUSTOM_VALID_PATH or None,
    "command": cmd,
}
(RUN_DIR / "run_plan.json").write_text(json.dumps(run_plan, indent=2), encoding="utf-8")
print(json.dumps(run_plan, indent=2))
print("Shell command:")
print(" \
".join(shlex.quote(part) for part in cmd))


In [ ]:
import os
import subprocess
import time

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"

with LOG_PATH.open("a", encoding="utf-8") as log_fp:
    log_fp.write(f"
===== RUN START {time.strftime('%Y-%m-%d %H:%M:%S')} =====
")
    proc = subprocess.Popen(
        cmd,
        cwd="/content/planner_colab_bundle",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        log_fp.write(line)
    ret = proc.wait()
    log_fp.write(f"===== RUN END code={ret} {time.strftime('%Y-%m-%d %H:%M:%S')} =====
")

if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)


In [ ]:
latest = latest_checkpoint(RUN_DIR)
print({
    "run_dir": str(RUN_DIR),
    "latest_checkpoint": str(latest) if latest else None,
    "log_path": str(LOG_PATH),
})
get_ipython().system(f"du -sh {RUN_DIR}")
get_ipython().system(f"find {RUN_DIR} -maxdepth 2 -type f | sort | tail -n 40")


In [ ]:
import shutil
from google.colab import files

TARGET_PATH = latest_checkpoint(RUN_DIR) or RUN_DIR
ARCHIVE_BASE = RUN_DIR.parent / TARGET_PATH.name
archive_path = shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=str(TARGET_PATH))
print(f"Wrote {archive_path}")
if DOWNLOAD_FINAL_ZIP:
    files.download(archive_path)
else:
    print("Skipping browser download; archive is saved in Drive/output root.")
